# Collection Operations Report - 数据抽取

**版本**: v2.1  
**更新说明**: 添加 `natural_month_repay` 月度累计回款数据，用于月时序回收进度  
**日期**: 请根据需要修改 `dt_start` 和 `dt_end`

In [128]:
# 参数配置
dt_start = '2026-03-01'  # 开始日期
dt_end = '2026-03-31'    # 结束日期

In [129]:
from get_write_data_from_dataworks import run_sql
import pandas as pd

---

## 1. TL Data - 每日组维度指标

**源表**: `phl_anls.tmp_liujun_ana_11_agent_process_daily`  
**用途**: TL View 的日指标

In [130]:
sql=f'''
-- TL Data: 每日组维度（Group × Day）
-- 用于 TL View 的日指标
SELECT
  owner_bucket AS module,
  owner_group AS group_id,
  dt,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  COUNT(DISTINCT owner_name) AS ownercount,
  -- 在手案量
  SUM(owing_case_cnt) AS total_owing_case,
  -- 覆盖
  SUM(comm_times) AS total_comm_times,
  SUM(own_uncomm_case_cnt) AS total_uncomm_case,
  -- 拨打
  SUM(call_times) AS total_calls,
  SUM(call_connect_times) AS total_connect,
  -- 接通率
  CASE WHEN SUM(call_times) > 0 THEN SUM(call_connect_times) / SUM(call_times) END AS connect_rate
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0  -- 仅内催
  AND owner_bucket IN ('S0','S1','S2','M1','M2')
GROUP BY
  owner_bucket,
  owner_group,
  dt
'''
tl_data=run_sql(sql)
tl_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=2026040109203421gh6sxxg02l7&token=cDM0TFNhcmk1dCtDTXZpUkhoOTdrUGNBU1lNPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NzYyNzIzNCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDAxMDkyMDM0MjFnaDZzeHhnMDJsNyJdfV0sIlZlcnNpb24iOiIxIn0=


,module,group_id,dt,headcount,ownercount,total_owing_case,total_comm_times,total_uncomm_case,total_calls,total_connect,connect_rate
0,M1,M1-Large A,2026-03-02,11,12,1741,15986,18,16669,497,0.029816
1,M1,M1-Large A,2026-03-03,12,12,1789,12464,125,12893,388,0.030094
2,M1,M1-Large A,2026-03-04,12,12,1787,14996,50,15504,480,0.030960
3,M1,M1-Large A,2026-03-05,10,12,1759,15846,47,16422,499,0.030386
4,M1,M1-Large A,2026-03-06,12,12,1702,12850,62,13361,399,0.029863


---

## 2. STL Data - 每周模块维度指标

**源表**: `phl_anls.tmp_liujun_ana_11_agent_process_daily`  
**用途**: STL View 的周指标

In [131]:
sql=f'''
-- STL Data: 每周模块维度（Module × Week）
-- 用于 STL View 的周指标
SELECT
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END AS module,
      -- 周区间：周起始为周六
      CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ) AS week,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  -- 在手案量
  SUM(owing_case_cnt) AS total_owing_case,
  -- 覆盖
  SUM(comm_times) AS total_comm_times,
  -- 拨打
  SUM(call_times) AS total_calls,
  SUM(call_connect_times) AS total_connect
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1','M2')
GROUP BY
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END,
  CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
)
'''
stl_data=run_sql(sql)
stl_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260401092035896gzt9wwz0zw6&token=WG1DN3d1TVFQOFloUjB6TkdOWGtCd0tKSTFBPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NzYyNzIzNix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDAxMDkyMDM1ODk2Z3p0OXd3ejB6dzYiXX1dLCJWZXJzaW9uIjoiMSJ9


,module,week,headcount,total_owing_case,total_comm_times,total_calls,total_connect
0,M1,2026-02-28-2026-03-06,31,27351,193117,196323,5501
1,M1,2026-03-07-2026-03-13,31,32912,254298,258822,7171
2,M1,2026-03-14-2026-03-20,31,35620,236139,239692,7038
3,M1,2026-03-21-2026-03-27,30,35412,228289,231925,6573
4,M1,2026-03-28-2026-04-03,30,17903,130711,132819,3573


---

## 3. Agent Performance - Agent日明细

**源表**: `phl_anls.tmp_liujun_ana_11_agent_process_daily`  
**用途**: TL View drill-down

In [144]:
sql=f'''
-- Agent Performance: Agent日明细（Group × Agent × Day）
-- 用于 TL View  drill-down
SELECT
  owner_bucket AS module,
  owner_group AS group_id,
  owner_name AS agent_id,
  dt,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  -- 在手案量
  SUM(owing_case_cnt)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) AS total_owing_case,
  -- 覆盖
  sum(comm_times)/(sum(owing_case_cnt) - sum(own_uncomm_case_cnt)) cover_times,
  -- 拨打
  SUM(call_times)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) AS call_times,
  SUM(art_call_times)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) as art_call_times,
  SUM(call_connect_times)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) AS call_connect_times,
  -- 接通时长
  SUM(call_billsec)/60/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) as call_billmin,
  -- 单通时长（分钟）
  (SUM(call_billsec / 60) / (
    GREATEST(COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END), 1)
    * COUNT(DISTINCT dt)
  )) / (
    SUM(call_connect_times) / (
      GREATEST(COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END), 1)
      * COUNT(DISTINCT dt)
    )
  ) AS single_call_duration,
  -- 电话接通率
  CASE WHEN SUM(call_times) > 0 THEN SUM(call_connect_times) / SUM(call_times) END AS connect_rate
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1','M2')
  and call_8h_flag > 0
group by   owner_bucket,
  owner_group,
  owner_name,
  dt
ORDER BY
  owner_bucket, owner_group, owner_name, dt
'''
agent_performance=run_sql(sql)
agent_performance.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260402063138678gywzoap1al7&token=RmhTdjdyalZGT0dhMHpPM0FpR1E0eEwrV2Y0PSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NzcwMzQ5OCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDAyMDYzMTM4Njc4Z3l3em9hcDFhbDciXX1dLCJWZXJzaW9uIjoiMSJ9


,module,group_id,agent_id,dt,headcount,total_owing_case,cover_times,call_times,art_call_times,call_connect_times,call_billmin,single_call_duration,connect_rate
0,M1,M1-Large A,Ramos02,2026-03-02,1,146.0,9.198630,1343.0,475.0,37.0,72.250000,1.952703,0.027550
1,M1,M1-Large A,Ramos02,2026-03-03,1,150.0,6.885135,1019.0,322.0,30.0,16.866667,0.562222,0.029441
2,M1,M1-Large A,Ramos02,2026-03-04,1,151.0,8.893333,1335.0,471.0,45.0,26.250000,0.583333,0.033708
3,M1,M1-Large A,Ramos02,2026-03-05,1,148.0,10.689189,1582.0,560.0,43.0,23.683333,0.550775,0.027181
4,M1,M1-Large A,Ramos02,2026-03-06,1,142.0,8.584507,1220.0,455.0,34.0,21.950000,0.645588,0.027869


---

## 4. Group Performance - 组周明细

**源表**: `phl_anls.tmp_liujun_ana_11_agent_process_daily`  
**用途**: STL View drill-down

In [145]:
sql=f'''
-- Group Performance: 组周明细（Module × Group × Week）
-- 用于 STL View drill-down
SELECT
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END AS module,
  owner_group AS group_id,
  CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ) AS week,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  -- 在手案量
  SUM(owing_case_cnt)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) AS total_owing_case,
  -- 覆盖
  sum(comm_times)/(sum(owing_case_cnt) - sum(own_uncomm_case_cnt)) cover_times,
  -- 拨打
  SUM(call_times)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) AS call_times,
  SUM(art_call_times)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) as art_call_times,
  SUM(call_connect_times)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) AS call_connect_times,
  -- 接通时长
  SUM(call_billsec)/60/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) as call_billmin,
  -- 单通时长（分钟）
  (SUM(call_billsec / 60) / (
    GREATEST(COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END), 1)
    * COUNT(DISTINCT dt)
  )) / (
    SUM(call_connect_times) / (
      GREATEST(COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END), 1)
      * COUNT(DISTINCT dt)
    )
  ) AS single_call_duration,
  -- 电话接通率
  CASE WHEN SUM(call_times) > 0 THEN SUM(call_connect_times) / SUM(call_times) END AS connect_rate
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1','M2')
  and call_8h_flag=1
GROUP BY
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END,
  owner_group,
  CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ) 
'''
group_performance=run_sql(sql)
group_performance.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260402063227696gf105r51zw6&token=TTYxQS81dlpkUGZ0SEM2Q2Jma0NzWnZsTkJvPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NzcwMzU0Nyx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDAyMDYzMjI3Njk2Z2YxMDVyNTF6dzYiXX1dLCJWZXJzaW9uIjoiMSJ9


,module,group_id,week,headcount,total_owing_case,cover_times,call_times,art_call_times,call_connect_times,call_billmin,single_call_duration,connect_rate
0,M1,M1-Large A,2026-02-28-2026-03-06,12,138.983333,8.604611,1200.783333,490.433333,36.316667,33.716944,0.928415,0.030244
1,M1,M1-Large A,2026-03-07-2026-03-13,12,132.041667,9.980070,1353.138889,557.097222,38.930556,31.802778,0.816910,0.028771
2,M1,M1-Large A,2026-03-14-2026-03-20,12,155.277778,8.499538,1319.666667,535.458333,39.513889,34.250000,0.866784,0.029942
3,M1,M1-Large A,2026-03-21-2026-03-27,12,147.763889,8.368743,1218.319444,496.500000,38.486111,33.937037,0.881800,0.031590
4,M1,M1-Large A,2026-03-28-2026-04-03,12,146.222222,10.095907,1465.861111,588.944444,41.388889,35.952315,0.868647,0.028235


---

## 5. 日目标回款 + Achievement（Agent日粒度）

**源表**: `phl_anls.rpt_coll_repay_target_dly4`  
**用途**: Module Daily Trends、Risk Analysis、Achievement 计算  
**说明**: 此表已包含日粒度的目标回款数据（target_repay_principal），可直接计算 achievement。用于TL tab

In [134]:
sql=f'''
-- 日目标回款 + Achievement
-- 源表: phl_anls.rpt_coll_repay_target_dly4
-- 用途: Module Daily Trends、Risk Analysis
SELECT
  dt,
  owner_bucket,
  owner_group,
  owner_name,
  SUM(owing_principal) AS daily_owing_principal,
  SUM(repay_principal) / NULLIF(SUM(owing_principal), 0) AS daily_repay_rate,
  SUM(target_repay_principal) AS target_repay_principal,
  SUM(repay_principal) AS actual_repay_principal,
  CASE WHEN SUM(target_repay_principal) <= 0 THEN 0
       ELSE SUM(repay_principal) / SUM(target_repay_principal)
  END AS achieve_rate
FROM (
  SELECT
    *,
    CASE
      WHEN owner_bucket = 'S0' AND case_overdue_days = -3 THEN 'H3'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -2 THEN 'H2'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -1 THEN 'H1'
      WHEN owner_bucket = 'S0' AND case_overdue_days >= 0 THEN 'H0'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 1 THEN '<=1'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 5 THEN '2-5'
      WHEN owner_bucket = 'S1' AND case_overdue_days >= 6 THEN '6-7'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 8 THEN '<=8'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 12 THEN '9-12'
      WHEN owner_bucket = 'S2' AND case_overdue_days >= 13 THEN '13-15'
      WHEN owner_bucket = 'M1' AND case_overdue_days <= 16 THEN '<=16'
      WHEN owner_bucket = 'M1' AND case_overdue_days >= 17 THEN '17-25'
      WHEN owner_bucket = 'M1' THEN '26-30'
      ELSE case_overdue_days
    END AS case_stage,
    CASE
      WHEN case_owing_principal <= 1000 THEN '<=1000'
      WHEN case_owing_principal <= 2000 THEN '1000-2000'
      WHEN case_owing_principal <= 5000 THEN '2000-5000'
      WHEN case_owing_principal > 5000 THEN '>5000'
    END AS principal_stage
  FROM phl_anls.rpt_coll_repay_target_dly4
  WHERE dt >= '{dt_start}'
    AND case_overdue_days > -99
    AND owner_bucket IN ('S0', 'S1', 'S2', 'M1')
) t
WHERE owner_bucket IN ('M1', 'S1', 'S2', 'S0')
  AND TO_CHAR(dt, 'yyyyMMdd') BETWEEN REPLACE('{dt_start}', '-', '') AND REPLACE('{dt_end}', '-', '')
GROUP BY
  dt,
  owner_bucket,
  owner_group,
  owner_name
HAVING SUM(owing_principal) > 0
ORDER BY dt DESC, owner_bucket DESC, owner_group ASC
'''
daily_target_agent_repay=run_sql(sql)
daily_target_agent_repay.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260401092042265geox8p18hrs5&token=YXpsdzk4ZjlWaytnSkJhVVJ5bDB0bVAzajVVPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NzYyNzI0Mix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDAxMDkyMDQyMjY1Z2VveDhwMThocnM1Il19XSwiVmVyc2lvbiI6IjEifQ==


,dt,owner_bucket,owner_group,owner_name,daily_owing_principal,daily_repay_rate,target_repay_principal,actual_repay_principal,achieve_rate
0,2026-03-31,S2,S2-A-AJCAI,outer_AJCAI004,3340,1.608682634730538922,33.4,5373,160.868263473053892216
1,2026-03-31,S2,S2-A-AJCAI,outer_AJCAI18,19555,0.774124264893889031,195.55,15138,77.412426489388903094
2,2026-03-31,S2,S2-A-AJCAI,outer_AJCAI58,12559,1.692730312923003424,9446.731110079999999999,21259,2.250408077913415634
3,2026-03-31,S2,S2-A-AJCAI,outer_AJCAI15,4017,1.654468508837440876,45770.575865140000045771,6646,0.145202455384918099
4,2026-03-31,S2,S2-A-AJCAI,outer_AJCAI32,27492,0.823148552306125418,274.919999999999999999,22630,82.314855230612541831


## 5.2 日目标回款 + Achievement（组周粒度）

**源表**: `phl_anls.rpt_coll_repay_target_dly4`  
**用途**: STL tab下的Module Daily Trends、Risk Analysis、Achievement 计算  
**说明**: 此表已包含日粒度的目标回款数据（target_repay_principal），可直接计算 achievement

In [135]:
sql=f'''
-- 日目标回款 + Achievement
-- 源表: phl_anls.rpt_coll_repay_target_dly4
-- 用途: STL tab下的Module Daily Trends、Risk Analysis
SELECT
    CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ) AS week,
  owner_bucket,
  owner_group,
  owner_name,
  SUM(owing_principal) AS daily_owing_principal,
  SUM(repay_principal) / NULLIF(SUM(owing_principal), 0) AS daily_repay_rate,
  SUM(target_repay_principal) AS target_repay_principal,
  SUM(repay_principal) AS actual_repay_principal,
  CASE WHEN SUM(target_repay_principal) <= 0 THEN 0
       ELSE SUM(repay_principal) / SUM(target_repay_principal)
  END AS achieve_rate
FROM (
  SELECT
    *,
    CASE
      WHEN owner_bucket = 'S0' AND case_overdue_days = -3 THEN 'H3'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -2 THEN 'H2'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -1 THEN 'H1'
      WHEN owner_bucket = 'S0' AND case_overdue_days >= 0 THEN 'H0'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 1 THEN '<=1'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 5 THEN '2-5'
      WHEN owner_bucket = 'S1' AND case_overdue_days >= 6 THEN '6-7'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 8 THEN '<=8'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 12 THEN '9-12'
      WHEN owner_bucket = 'S2' AND case_overdue_days >= 13 THEN '13-15'
      WHEN owner_bucket = 'M1' AND case_overdue_days <= 16 THEN '<=16'
      WHEN owner_bucket = 'M1' AND case_overdue_days >= 17 THEN '17-25'
      WHEN owner_bucket = 'M1' THEN '26-30'
      ELSE case_overdue_days
    END AS case_stage,
    CASE
      WHEN case_owing_principal <= 1000 THEN '<=1000'
      WHEN case_owing_principal <= 2000 THEN '1000-2000'
      WHEN case_owing_principal <= 5000 THEN '2000-5000'
      WHEN case_owing_principal > 5000 THEN '>5000'
    END AS principal_stage
  FROM phl_anls.rpt_coll_repay_target_dly4
  WHERE dt >= '{dt_start}'
    AND case_overdue_days > -99
    AND owner_bucket IN ('S0', 'S1', 'S2', 'M1')
) t
WHERE owner_bucket IN ('M1', 'S1', 'S2', 'S0')
  AND TO_CHAR(dt, 'yyyyMMdd') BETWEEN REPLACE('{dt_start}', '-', '') AND REPLACE('{dt_end}', '-', '')
GROUP BY
    CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ),
  owner_bucket,
  owner_group,
  owner_name
HAVING SUM(owing_principal) > 0
ORDER BY week DESC, owner_bucket DESC, owner_group ASC
'''
daily_target_group_repay=run_sql(sql)
daily_target_group_repay.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260401092046474g5y9wwz0zw6&token=bHRVbThPK0xUK2xMdTJ2c2tjQkpCcGdrRU9FPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NzYyNzI0Nix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDAxMDkyMDQ2NDc0ZzV5OXd3ejB6dzYiXX1dLCJWZXJzaW9uIjoiMSJ9


,week,owner_bucket,owner_group,owner_name,daily_owing_principal,daily_repay_rate,target_repay_principal,actual_repay_principal,achieve_rate
0,2026-03-28-2026-04-03,S2,S2-A-AJCAI,outer_AJCAI004,1064007,0.022642708177671763,27308.882885859999917936,24092,0.882203790638186876
1,2026-03-28-2026-04-03,S2,S2-A-AJCAI,outer_AJCAI15,1223681.84,0.014412242973222517,154299.551279795116677835,17636,0.114297156755953319
2,2026-03-28-2026-04-03,S2,S2-A-AJCAI,outer_AJCAI32,1107916,0.033458312724069334,27710.875052682000010549,37069,1.33770586203167456
3,2026-03-28-2026-04-03,S2,S2-A-AJCAI,outer_AJCAI62,1113857,0.045746446806008312,37279.191741414166738286,50955,1.366848303832540351
4,2026-03-28-2026-04-03,S2,S2-A-AJCAI,outer_AJCAI006,1126827,0.016131136367871909,90622.045768707300267066,18177,0.200580331704194434


---

---

## 6. Natural Month Repay - 月度累计回款

**源表**: `tmp_maoruochen_phl_repay_natural_day_daily`  
**用途**: 月时序回收进度（Month Trends）  
**说明**: `dt` 为业务后一天，`substr(dt,9,2)='01'` 表示"上月月末累计状态"

In [136]:
# Natural Month Repay: 月度累计回款
# dt 为业务后一天，因此 substr(dt,9,2)='01' 表示"上月月末累计状态"
sql=f'''
SELECT
  dt_biz,
  agent_bucket,
  group_name,
  SUM(repay_principal) AS repay_principal,
  SUM(start_owing_principal) AS start_owing_principal,
  SUM(repay_principal) / SUM(start_owing_principal) as repay_rate
FROM phl_anls.tmp_maoruochen_phl_repay_natural_day_daily
WHERE data_level = '4.组别层级'  
  AND TO_CHAR(dt_biz, 'yyyyMM') = '202603'
  AND group_name is not null 
GROUP BY dt_biz, agent_bucket, group_name
;
'''
natural_month_repay=run_sql(sql)
natural_month_repay.head(100)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260401092051872gcnw1dcx9l7&token=cGVWWHNQQ3BlVVVpZGM4WCsrZUg4SE9mdFdjPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NzYyNzI1MSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDAxMDkyMDUxODcyZ2NudzFkY3g5bDciXX1dLCJWZXJzaW9uIjoiMSJ9


,dt_biz,agent_bucket,group_name,repay_principal,start_owing_principal,repay_rate
0,2026-03-01,M1,Target,0.00893237,1,0.00893237
1,2026-03-02,M1,Target,0.01170479,1,0.01170479
2,2026-03-03,M1,Target,0.01692765,1,0.01692765
3,2026-03-04,M1,Target,0.02175289,1,0.02175289
4,2026-03-05,M1,Target,0.02640341,1,0.02640341
...,...,...,...,...,...,...
95,2026-03-03,M1_Small,Target,0.02693653,1,0.02693653
96,2026-03-04,M1_Small,Target,0.03343004,1,0.03343004
97,2026-03-05,M1_Small,Target,0.03945202,1,0.03945202
98,2026-03-06,M1_Small,Target,0.04525791,1,0.04525791


---

## 7. PTP Group Data - 人日PTP明细

**源表**: `phl_anls.tmp_zyt_agent_ptp`  
**用途**: TL View drill-down PTP履约率  
**粒度**: `owner_name × dt`

In [137]:
sql=f'''
-- PTP 承诺还款 - 人日明细（Agent × Day）
-- 用于 TL View drill-down
SELECT
  dt,
  owner_group,
  owner_name,
  SUM(ptp_case) AS ptp_case_cnt,
  SUM(ptp_today_case) AS ptp_today_case,
  SUM(today_actual_repay_case) AS today_actual_repay_case,
  CASE WHEN SUM(ptp_today_case) > 0
       THEN SUM(today_actual_repay_case) / SUM(ptp_today_case)
       ELSE 0
  END AS today_ptp_repay_rate
FROM phl_anls.tmp_zyt_agent_ptp
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
GROUP BY dt, owner_group, owner_name
ORDER BY dt, owner_group ASC
'''
ptp_agent_data = run_sql(sql)
ptp_agent_data.head(10)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260401092057941gutx8p18hrs5&token=KytsR3orbXRHZ3JNeEZzRE1PRERkVzl4ZlJJPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NzYyNzI1OCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDAxMDkyMDU3OTQxZ3V0eDhwMThocnM1Il19XSwiVmVyc2lvbiI6IjEifQ==


,dt,owner_group,owner_name,ptp_case_cnt,ptp_today_case,today_actual_repay_case,today_ptp_repay_rate
0,2026-03-01,None,garcia03,0,0,0,0.0
1,2026-03-01,M4+-B-AJCAI,outer_AJCAI75,0,0,0,0.0
2,2026-03-01,M4+-B-AJCAI,outer_AJCAI73,0,0,0,0.0
3,2026-03-01,M4+-B-AJCAI,outer_AJCAI71,0,0,0,0.0
4,2026-03-01,M4+-B-AJCAI,outer_AJCAI72,0,0,0,0.0
5,2026-03-01,M4+-B-AJCAI,outer_AJCAI74,0,0,0,0.0
6,2026-03-01,Cluster A-Nocall,S1-NOCA,0,0,0,0.0
7,2026-03-01,Cluster A-Nocall,S0-NOCA,0,0,0,0.0
8,2026-03-01,L0 Bucket,suderiond,0,0,0,0.0
9,2026-03-01,L0 Bucket,villacarlosjb,0,0,0,0.0


---

## 8. PTP Group Data - 组周PTP明细

**源表**: `phl_anls.tmp_zyt_agent_ptp`  
**用途**: STL View drill-down PTP履约率  
**粒度**: `owner_group × week`

In [138]:
sql=f'''
-- PTP 承诺还款 - 组周明细（Group × Week）
-- 用于 STL View drill-down
SELECT
    CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ) AS week,
  owner_group,
  SUM(ptp_case) AS ptp_case_cnt,
  SUM(ptp_today_case) AS ptp_today_case,
  SUM(today_actual_repay_case) AS today_actual_repay_case,
  CASE WHEN SUM(ptp_today_case) > 0
       THEN SUM(today_actual_repay_case) / SUM(ptp_today_case)
       ELSE 0
  END AS today_ptp_repay_rate
FROM phl_anls.tmp_zyt_agent_ptp
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
GROUP BY   CONCAT(
  -- 周起始：周六
  TO_CHAR(
    DATEADD(
      TO_DATE(dt, 'yyyy-MM-dd'), 
      -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
      'dd'
    ), 
    'yyyy-MM-dd'
  ),
  '-',
  -- 周结束：周五
  TO_CHAR(
    DATEADD(
      TO_DATE(dt, 'yyyy-MM-dd'), 
      ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
      'dd'
    ), 
    'yyyy-MM-dd'
  )
), owner_group
ORDER BY week DESC, owner_group ASC
'''
ptp_group_data = run_sql(sql)
ptp_group_data.head()

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260401092100514gh2awwz0zw6&token=MDF0ZnpnYS8zMWNYWlBqaFhKSGY0b1VIUUpjPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NzYyNzI2MCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDAxMDkyMTAwNTE0Z2gyYXd3ejB6dzYiXX1dLCJWZXJzaW9uIjoiMSJ9


,week,owner_group,ptp_case_cnt,ptp_today_case,today_actual_repay_case,today_ptp_repay_rate
0,2026-03-28-2026-04-03,None,0,0,0,0.000000
1,2026-03-28-2026-04-03,M4+-B-AJCAI,4,4,1,0.250000
2,2026-03-28-2026-04-03,L0 Bucket,129,68,60,0.882353
3,2026-03-28-2026-04-03,L1 Bucket,132,99,60,0.606061
4,2026-03-28-2026-04-03,L2 Bucket,12,4,1,0.250000


---

## 9. 呼损率 - 人日明细

**源表**: `phl_anls.tmp_phl_zyt_20_lark_genesys_daily`  
**用途**: TL View drill-down 呼损率监控  
**筛选**: `call_type = '3'`（仅一键外呼）  
**粒度**: `owner_name × dt`

In [139]:
sql=f'''
-- 呼损率数据 - 人日明细（Agent × Day）
-- 用于 STL View drill-down
-- 仅统计一键外呼 (call_type = '3')
SELECT
  dt,
  group_name,
  owner_name,
  SUM(call_loss) / (SUM(call_loss) + SUM(connect_times)) AS call_loss_rate,
  SUM(call_loss) AS call_loss,
  SUM(connect_times) AS connect_times
FROM phl_anls.tmp_phl_zyt_20_lark_genesys_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND CAST(call_type AS STRING) = '3'
GROUP BY dt, group_name, owner_name
ORDER BY dt DESC, group_name ASC
'''
call_loss_agent_data = run_sql(sql)
call_loss_agent_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260401092102476gbrw1dcx9l7&token=SmJnRlBQcStIZVM1K2RqRFFvRTQ3cENXbCs0PSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NzYyNzI2Mix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDAxMDkyMTAyNDc2Z2JydzFkY3g5bDciXX1dLCJWZXJzaW9uIjoiMSJ9


,dt,group_name,owner_name,call_loss_rate,call_loss,connect_times
0,2026-03-31,M4+-B-AJCAI,outer_AJCAI75,0.468750,15,17
1,2026-03-31,M4+-B-AJCAI,outer_AJCAI71,0.230769,6,20
2,2026-03-31,L0 Bucket,jessie,0.250000,6,18
3,2026-03-31,L0 Bucket,ramosjm,0.619048,13,8
4,2026-03-31,L0 Bucket,suderiond,0.472222,17,19


---

## 10. 呼损率 - 组周明细

**源表**: `phl_anls.tmp_phl_zyt_20_lark_genesys_daily`  
**用途**: STL View drill-down 呼损率监控  
**筛选**: `call_type = '3'`（仅一键外呼）  
**粒度**: `group_name × week`

In [140]:
sql=f'''
-- 呼损率数据 - 组周明细（Group × Week）
-- 用于 STL View drill-down
-- 仅统计一键外呼 (call_type = '3')
SELECT
CONCAT(
  -- 周起始：周六
  TO_CHAR(
    DATEADD(
      TO_DATE(dt, 'yyyy-MM-dd'), 
      -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
      'dd'
    ), 
    'yyyy-MM-dd'
  ),
  '-',
  -- 周结束：周五
  TO_CHAR(
    DATEADD(
      TO_DATE(dt, 'yyyy-MM-dd'), 
      ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
      'dd'
    ), 
    'yyyy-MM-dd'
  )
) AS week,
  group_name,
  SUM(call_loss) / (SUM(call_loss) + SUM(connect_times)) AS call_loss_rate,
  SUM(call_loss) AS call_loss,
  SUM(connect_times) AS connect_times
FROM phl_anls.tmp_phl_zyt_20_lark_genesys_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND CAST(call_type AS STRING) = '3'
GROUP BY   CONCAT(
  -- 周起始：周六
  TO_CHAR(
    DATEADD(
      TO_DATE(dt, 'yyyy-MM-dd'), 
      -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
      'dd'
    ), 
    'yyyy-MM-dd'
  ),
  '-',
  -- 周结束：周五
  TO_CHAR(
    DATEADD(
      TO_DATE(dt, 'yyyy-MM-dd'), 
      ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
      'dd'
    ), 
    'yyyy-MM-dd'
  )
), group_name
ORDER BY week DESC, group_name ASC
'''
call_loss_group_data = run_sql(sql)
call_loss_group_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=2026040109210569gesw1dcx9l7&token=NzVRRmVLMlBZa0UzaUxmbWZIcjJKRGJSS01NPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NzYyNzI2NSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDAxMDkyMTA1NjlnZXN3MWRjeDlsNyJdfV0sIlZlcnNpb24iOiIxIn0=


,week,group_name,call_loss_rate,call_loss,connect_times
0,2026-03-28-2026-04-03,M4+-B-AJCAI,0.346154,36,68
1,2026-03-28-2026-04-03,L0 Bucket,0.389189,144,226
2,2026-03-28-2026-04-03,L1 Bucket,0.368286,288,494
3,2026-03-28-2026-04-03,L2 Bucket,0.410714,161,231
4,2026-03-28-2026-04-03,L3 Bucket,0.396154,103,157


## 输出说明

| Cell | 输出变量名 | 用途 | drill-down 层级 |
|------|-----------|------|----------------|
| 1 | `tl_data` | TL每日指标 | — |
| 2 | `stl_data` | STL每周指标 | — |
| 3 | `agent_performance` | Agent日明细 | TL drill-down |
| 4 | `group_performance` | 组周明细 | STL drill-down |
| 5 | `daily_target_agent_repay` | Agent日目标回款（含achievement） | TL |
| 6 | `daily_target_agent_repay` | 组周目标回款（含achievement） | STL |
| 7 | `natural_month_repay` | 月度累计回款（Month Trends） | — |
| 8 | `ptp_agent_data` | Agent日PTP履约明细 | TL drill-down |
| 9 | `ptp_group_data` | 组周PTP履约明细 | STL drill-down |
| 10 | `call_loss_data` | Agent日呼损率 | TL drill-down |
| 11 | `call_loss_data` | 组周呼损率 | STL drill-down |

In [ ]:
# 输出为 Excel（推荐）
with pd.ExcelWriter('260318_output_automation_v3.xlsx') as writer:
    tl_data.to_excel(writer, sheet_name='tl_data', index=True)
    stl_data.to_excel(writer, sheet_name='stl_data', index=True)
    agent_performance.to_excel(writer, sheet_name='agent_performance', index=True)
    group_performance.to_excel(writer, sheet_name='group_performance', index=True)
    daily_target_agent_repay.to_excel(writer, sheet_name='daily_target_agent_repay', index=True)
    daily_target_group_repay.to_excel(writer, sheet_name='daily_target_group_repay', index=True)
    natural_month_repay.to_excel(writer, sheet_name='natural_month_repay', index=True)
    ptp_agent_data.to_excel(writer, sheet_name='ptp_agent_data', index=True)
    ptp_group_data.to_excel(writer, sheet_name='ptp_group_data', index=True)
    call_loss_agent_data.to_excel(writer, sheet_name='call_loss_agent_data', index=True)
    call_loss_group_data.to_excel(writer, sheet_name='call_loss_group_data', index=True)